# 第 6 章 练习题答案

> 精选 2 道核心练习，巩固分类微调理解。

## 练习 6.1：为什么用最后一个 token 做分类？

**题目**：分类微调时为什么取最后一个 token 的输出，而不是第一个或平均？用代码验证因果注意力的信息流。

In [ ]:
import torch
from src.gpt import GPTModel, GPT_CONFIG_124M

cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 64, "n_layers": 2, "n_heads": 4, "context_length": 8})
torch.manual_seed(123)
model = GPTModel(cfg)
model.out_head = torch.nn.Linear(cfg["emb_dim"], 2)  # 分类头

x = torch.tensor([[1, 2, 3, 4]])  # 4 个 token
model.eval()
with torch.no_grad():
    out = model(x)  # [1, 4, 2]

print(f"输入 {x.shape}，输出 {out.shape}")
print(f"\n各位置 token 的分类 logits（取最后一个位置做分类）:")
for i in range(4):
    print(f"  位置 {i}: {out[0, i].tolist()}")

print(f"\n用最后一个 token: {out[0, -1].tolist()}")
print("\n💡 因果注意力下，位置 i 只能看到 0..i。")
print("   最后一个 token 看过全部内容，信息最全 → 最适合做整体分类。")
print("   （详见 ch06/bonus/01 的对比实验：最后token 100% vs 首token 69%）")

## 练习 6.2：冻结策略对比

**题目**：冻结 backbone vs 全量微调，可训练参数差多少？

In [ ]:
import torch.nn as nn
from src.gpt import GPTModel, GPT_CONFIG_124M

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build(freeze):
    cfg = dict(GPT_CONFIG_124M)
    cfg.update({"emb_dim": 64, "n_layers": 2, "n_heads": 4, "context_length": 8})
    torch.manual_seed(0)
    m = GPTModel(cfg)
    m.out_head = nn.Linear(cfg["emb_dim"], 2)
    if freeze:
        for p in m.parameters(): p.requires_grad = False
        for p in m.trf_blocks[-1].parameters(): p.requires_grad = True
        for p in m.final_norm.parameters(): p.requires_grad = True
        for p in m.out_head.parameters(): p.requires_grad = True
    return m

full = build(freeze=False)
frozen = build(freeze=True)
total = sum(p.numel() for p in full.parameters())
print(f"{'策略':<16} {'可训练参数':>12} {'占比':>8}")
print("-" * 38)
print(f"{'全量微调':<16} {count_trainable(full):>12,} {100*count_trainable(full)/total:>7.1f}%")
print(f"{'冻结backbone':<16} {count_trainable(frozen):>12,} {100*count_trainable(frozen)/total:>7.1f}%")
print("\n💡 冻结策略让可训练参数大幅减少，省显存、防小数据过拟合。")
print("   效果通常接近全量微调（尤其数据量小时）。")